### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

freq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413169 entries, 0 to 413168
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   PolicyID   413169 non-null  category
 1   ClaimNb    413169 non-null  int32   
 2   Exposure   413169 non-null  float64 
 3   Power      413169 non-null  category
 4   CarAge     413169 non-null  int32   
 5   DriverAge  413169 non-null  int32   
 6   Brand      413169 non-null  category
 7   Gas        413169 non-null  category
 8   Region     413169 non-null  category
 9   Density    413169 non-null  int32   
dtypes: category(5), float64(1), int32(4)
memory usage: 31.9 MB


In [3]:
X=freq[['Exposure', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density']]
y=freq['ClaimNb']

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True, dtype=int)
display(X)

# Split 75% / 25%
X_train, X_test, y_train, y_test_ClaimN = train_test_split(X, y, test_size=0.25, random_state=42)


,Exposure,CarAge,DriverAge,Density,Brand_Japanese (except Nissan) or Korean,"Brand_Mercedes, Chrysler or BMW","Brand_Opel, General Motors or Ford",Brand_other,"Brand_Renault, Nissan or Citroen","Brand_Volkswagen, Audi, Skoda or Seat",Gas_Regular,Region_Basse-Normandie,Region_Bretagne,Region_Centre,Region_Haute-Normandie,Region_Ile-de-France,Region_Limousin,Region_Nord-Pas-de-Calais,Region_Pays-de-la-Loire,Region_Poitou-Charentes
0,0.090000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0.840000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.520000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
3,0.450000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
4,0.150000,0,41,60,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413164,0.002740,0,29,2471,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413165,0.005479,0,29,5360,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
413166,0.005479,0,49,5360,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413167,0.002740,0,41,9850,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0




#### Principe de la fonction
La fonction `stepwise_selection` est utilisée pour effectuer une sélection de variables explicatives dans un modèle statistique, en se basant sur des critères d'information comme l'AIC (Akaike Information Criterion) ou le BIC (Bayesian Information Criterion). Voici les étapes principales de cette fonction :

1. **Initialisation** :
    - Toutes les variables explicatives présentes dans `X` sont incluses dans le modèle initial.
    - Un modèle est ajusté avec toutes les variables, et la valeur du critère choisi (AIC ou BIC) est calculée.

2. **Itérations** :
    - À chaque itération, la fonction teste le retrait de chaque variable explicative une par une.
    - Pour chaque sous-ensemble de variables (obtenu en retirant une variable), un modèle est ajusté, et la valeur du critère est calculée.

3. **Sélection de la variable à retirer** :
    - La variable dont le retrait entraîne la plus faible valeur du critère est identifiée.
    - Si cette nouvelle valeur est inférieure à celle du modèle actuel, la variable est retirée du modèle, et le processus continue.
    - Sinon, l'algorithme s'arrête, car retirer d'autres variables n'améliore plus le critère.

4. **Résultat final** :
    - Le modèle final est ajusté avec les variables sélectionnées.
    - La fonction retourne le modèle final et la liste des variables sélectionnées.


In [4]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 309876 entries, 408079 to 121958
Data columns (total 20 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   Exposure                                  309876 non-null  float64
 1   CarAge                                    309876 non-null  int32  
 2   DriverAge                                 309876 non-null  int32  
 3   Density                                   309876 non-null  int32  
 4   Brand_Japanese (except Nissan) or Korean  309876 non-null  int32  
 5   Brand_Mercedes, Chrysler or BMW           309876 non-null  int32  
 6   Brand_Opel, General Motors or Ford        309876 non-null  int32  
 7   Brand_other                               309876 non-null  int32  
 8   Brand_Renault, Nissan or Citroen          309876 non-null  int32  
 9   Brand_Volkswagen, Audi, Skoda or Seat     309876 non-null  int32  
 10  Gas_Regular         

In [5]:
# Fonction pour effectuer la sélection de variables basée sur AIC et BIC
def stepwise_selection(X, y, family, criterion='AIC'):
    """
    Effectue une sélection de variables en utilisant AIC ou BIC.
    
    Parameters:
        X (pd.DataFrame): Variables explicatives.
        y (pd.Series): Variable cible.
        family: Famille de distribution (ex: sm.families.Poisson()).
        criterion (str): Critère de sélection ('AIC' ou 'BIC').
    
    Returns:
        result (GLMResultsWrapper): Résultat du modèle final.
        selected_features (list): Liste des variables sélectionnées.
    """
    selected_features = list(X.columns)
    current_model = sm.GLM(y, sm.add_constant(X[selected_features]), family=family).fit()
    current_criterion = getattr(current_model, criterion.lower())
    
    while True:
        criteria = []
        for feature in selected_features:
            features_to_test = [f for f in selected_features if f != feature]
            model = sm.GLM(y, sm.add_constant(X[features_to_test]), family=family).fit()
            model_criterion = getattr(model, criterion.lower())
            criteria.append((model_criterion, feature))
            
            # Afficher la valeur du critère pour chaque modèle
            #print(f"Modèle'{features_to_test}': {criterion} = {model_criterion}")
        
        criteria.sort()
        best_criterion, worst_feature = criteria[0]
        
        if best_criterion < current_criterion:
            selected_features.remove(worst_feature)
            current_criterion = best_criterion
        else:
            break
    
    final_model = sm.GLM(y, sm.add_constant(X[selected_features]), family=family).fit()
    return final_model, selected_features



In [6]:


# Appliquer la sélection de variables
final_model_AIC_ClaimN, selected_features_AIC_ClaimN = stepwise_selection(X_train, y_train, sm.families.Poisson(), criterion='AIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_AIC_ClaimN)
display(final_model_AIC_ClaimN.summary())

Variables sélectionnées : ['Exposure', 'CarAge', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Opel, General Motors or Ford', 'Brand_Renault, Nissan or Citroen', 'Gas_Regular', 'Region_Bretagne', 'Region_Haute-Normandie', 'Region_Ile-de-France', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309860
Model Family:                 Poisson   Df Model:                           15
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50426.
Date:                Mon, 01 Dec 2025   Deviance:                       77525.
Time:                        22:05:01   Pearson chi2:                 3.25e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.007964
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                       -3.5398      0.042    -84.220      0.000      -3.622      -3.457
Exposure                                     1.2070      0.028     42.478      0.000       1.151       1.263
CarAge                                      -0.0090      0.002     -4.990      0.000      -0.013      -0.005
DriverAge                                   -0.0075      0.001    -11.198      0.000      -0.009      -0.006
Density                                   1.468e-05   2.24e-06      6.562      0.000    1.03e-05    1.91e-05
Brand_Japanese (except Nissan) or Korean    -0.3629      0.035    -10.313      0.000      -0.432      -0.294
Brand_Opel, General Motors or Ford           0.0589      0.034      1.730      0.084      -0.008       0.126
Brand_Renault, Nissan or Citroen            -0.0845      0.023     -3.606      0.000      -0.130      -0.039
Gas_Regular                                 -0.1104      0.019     -5.887      0.000      -0.147      -0.074
Region_Bretagne                              0.0822      0.030      2.778      0.005       0.024       0.140
Region_Haute-Normandie                      -0.2063      0.081     -2.532      0.011      -0.366      -0.047
Region_Ile-de-France                         0.1387      0.035      4.001      0.000       0.071       0.207
Region_Limousin                              0.3096      0.081      3.817      0.000       0.151       0.469
Region_Nord-Pas-de-Calais                    0.0807      0.041      1.990      0.047       0.001       0.160
Region_Pays-de-la-Loire                      0.1074      0.032      3.384      0.001       0.045       0.170
Region_Poitou-Charentes                      0.0974      0.043      2.254      0.024       0.013       0.182
============================================================================================================
"""

In [7]:
import warnings

# Appliquer la sélection de variables
warnings.filterwarnings("ignore")

final_model_BIC_ClaimN, selected_features_BIC_ClaimN = stepwise_selection(X_train, y_train, sm.families.Poisson(), criterion='BIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_BIC_ClaimN)
display(final_model_BIC_ClaimN.summary())

Variables sélectionnées : ['Exposure', 'CarAge', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Renault, Nissan or Citroen', 'Gas_Regular']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309868
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50453.
Date:                Mon, 01 Dec 2025   Deviance:                       77578.
Time:                        22:07:47   Pearson chi2:                 3.25e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.007796
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                       -3.4690      0.038    -90.315      0.000      -3.544      -3.394
Exposure                                     1.2073      0.028     42.882      0.000       1.152       1.262
CarAge                                      -0.0093      0.002     -5.186      0.000      -0.013      -0.006
DriverAge                                   -0.0076      0.001    -11.541      0.000      -0.009      -0.006
Density                                   1.857e-05   1.81e-06     10.247      0.000     1.5e-05    2.21e-05
Brand_Japanese (except Nissan) or Korean    -0.3525      0.032    -10.904      0.000      -0.416      -0.289
Brand_Renault, Nissan or Citroen            -0.1093      0.020     -5.360      0.000      -0.149      -0.069
Gas_Regular                                 -0.1067      0.019     -5.700      0.000      -0.143      -0.070
============================================================================================================
"""

In [8]:
selected_features_AIC_ClaimN

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Opel, General Motors or Ford',
 'Brand_Renault, Nissan or Citroen',
 'Gas_Regular',
 'Region_Bretagne',
 'Region_Haute-Normandie',
 'Region_Ile-de-France',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [9]:
selected_features_BIC_ClaimN

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Renault, Nissan or Citroen',
 'Gas_Regular']

In [10]:
# Ajouter une constante à X_test pour les deux modèles
X_test_const_AIC_ClaimN = sm.add_constant(X_test[selected_features_AIC_ClaimN])
X_test_const_BIC_ClaimN = sm.add_constant(X_test[selected_features_BIC_ClaimN])

# Prédictions pour les deux modèles
y_pred_test_AIC_ClaimN = final_model_AIC_ClaimN.predict(X_test_const_AIC_ClaimN)
y_pred_test_BIC_ClaimN = final_model_BIC_ClaimN.predict(X_test_const_BIC_ClaimN)

# Calcul des MSE pour les deux modèles
mse_AIC = mean_squared_error(y_test_ClaimN, y_pred_test_AIC_ClaimN)
mse_BIC = mean_squared_error(y_test_ClaimN, y_pred_test_BIC_ClaimN)

print("MSE pour le modèle AIC ClaimN :", mse_AIC)
print("MSE pour le modèle BIC ClaimN :", mse_BIC)


MSE pour le modèle AIC ClaimN : 0.04211466893706517
MSE pour le modèle BIC ClaimN : 0.04211065684750139


In [11]:
y_pred_test_AIC_ClaimN.mean()

0.039014659618990255

In [28]:

y_pred_test_BIC_ClaimN.mean()

0.03900624982056504

In [29]:
y_test_ClaimN.mean()

0.03978004317814373

In [32]:
count_claim_nb_ge_1 = (y_test_ClaimN >= 1).sum()
print("Nombre de ClaimNb >= 1 :", count_claim_nb_ge_1)

Nombre de ClaimNb >= 1 : 3902


## Coût des sinistres

In [13]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

merged = pd.merge(freq, sev, on='PolicyID', how='left')
num_cols = merged.select_dtypes(include=['float', 'int']).columns
merged[num_cols] = merged[num_cols].fillna(0)
#display(merged)


In [14]:
X=merged[['Exposure', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density']]
y=merged['ClaimAmount']

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True, dtype=int)
display(X)

# Split 75% / 25%
X_train, X_test, y_train, y_test_ClaimA = train_test_split(X, y, test_size=0.25, random_state=42)

,Exposure,CarAge,DriverAge,Density,Brand_Japanese (except Nissan) or Korean,"Brand_Mercedes, Chrysler or BMW","Brand_Opel, General Motors or Ford",Brand_other,"Brand_Renault, Nissan or Citroen","Brand_Volkswagen, Audi, Skoda or Seat",Gas_Regular,Region_Basse-Normandie,Region_Bretagne,Region_Centre,Region_Haute-Normandie,Region_Ile-de-France,Region_Limousin,Region_Nord-Pas-de-Calais,Region_Pays-de-la-Loire,Region_Poitou-Charentes
0,0.090000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0.840000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.520000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
3,0.450000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
4,0.150000,0,41,60,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413955,0.002740,0,29,2471,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413956,0.005479,0,29,5360,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
413957,0.005479,0,49,5360,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413958,0.002740,0,41,9850,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0


In [15]:
# Appliquer la sélection de variables
final_model_AIC_ClaimA, selected_features_AIC_ClaimA = stepwise_selection(X_train, y_train, sm.families.Gamma(), criterion='AIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_AIC_ClaimA)
display(final_model_AIC_ClaimA.summary())

Variables sélectionnées : ['Exposure', 'CarAge', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Mercedes, Chrysler or BMW', 'Brand_Opel, General Motors or Ford', 'Brand_other', 'Brand_Renault, Nissan or Citroen', 'Brand_Volkswagen, Audi, Skoda or Seat', 'Gas_Regular', 'Region_Basse-Normandie', 'Region_Bretagne', 'Region_Centre', 'Region_Haute-Normandie', 'Region_Ile-de-France', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            ClaimAmount   No. Observations:               310470
Model:                            GLM   Df Residuals:                   310449
Model Family:                   Gamma   Df Model:                           20
Link Function:           InversePower   Scale:                          346.54
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Mon, 01 Dec 2025   Deviance:                   2.1487e+07
Time:                        22:09:31   Pearson chi2:                 1.08e+08
No. Iterations:                    19   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                        0.0117      0.001     12.260      0.000       0.010       0.014
Exposure                                     0.0028      0.000     27.647      0.000       0.003       0.003
CarAge                                       0.0004   1.38e-05     27.809      0.000       0.000       0.000
DriverAge                                    0.0002   8.01e-06     27.823      0.000       0.000       0.000
Density                                   3.231e-07   1.25e-08     25.830      0.000    2.99e-07    3.48e-07
Brand_Japanese (except Nissan) or Korean     0.0015      0.000      7.676      0.000       0.001       0.002
Brand_Mercedes, Chrysler or BMW             -0.0005      0.000     -2.942      0.003      -0.001      -0.000
Brand_Opel, General Motors or Ford          -0.0013      0.000     -6.982      0.000      -0.002      -0.001
Brand_other                                 -0.0050      0.000    -18.049      0.000      -0.006      -0.004
Brand_Renault, Nissan or Citroen            -0.0095      0.000    -25.043      0.000      -0.010      -0.009
Brand_Volkswagen, Audi, Skoda or Seat       -0.0152      0.001    -26.693      0.000      -0.016      -0.014
Gas_Regular                                 -0.0035      0.000    -27.687      0.000      -0.004      -0.003
Region_Basse-Normandie                      -0.0173      0.001    -16.999      0.000      -0.019      -0.015
Region_Bretagne                             -0.0038      0.001     -4.291      0.000      -0.006      -0.002
Region_Centre                               -0.0087      0.001     -9.566      0.000      -0.010      -0.007
Region_Haute-Normandie                       0.0102      0.006      1.672      0.095      -0.002       0.022
Region_Ile-de-France                        -0.0013      0.001     -1.494      0.135      -0.003       0.000
Region_Limousin                              0.0011      0.005      0.218      0.828      -0.009       0.011
Region_Nord-Pas-de-Calais                    0.0019      0.002      0.843      0.399      -0.003       0.006
Region_Pays-de-la-Loire                      0.0005      0.001      0.395      0.693      -0.002       0.003
Region_Poitou-Charentes                     -0.0001      0.001     -0.104      0.917      -0.002       0.002
============================================================================================================
"""

In [16]:
import warnings

# Appliquer la sélection de variables
warnings.filterwarnings("ignore")

final_model_BIC_ClaimA, selected_features_BIC_ClaimA = stepwise_selection(X_train, y_train, sm.families.Gamma(), criterion='BIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_BIC_ClaimA)
display(final_model_BIC_ClaimA.summary())

Variables sélectionnées : ['Density', 'Brand_Opel, General Motors or Ford', 'Gas_Regular', 'Region_Centre', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            ClaimAmount   No. Observations:               310470
Model:                            GLM   Df Residuals:                   310461
Model Family:                   Gamma   Df Model:                            8
Link Function:           InversePower   Scale:                          1550.7
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Mon, 01 Dec 2025   Deviance:                   2.1450e+07
Time:                        22:18:36   Pearson chi2:                 4.81e+08
No. Iterations:                    19   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                  0.0143      0.002      7.286      0.000       0.010       0.018
Density                             9.585e-08   2.31e-07      0.415      0.678   -3.57e-07    5.49e-07
Brand_Opel, General Motors or Ford     0.0013      0.003      0.420      0.674      -0.005       0.007
Gas_Regular                           -0.0022      0.002     -1.356      0.175      -0.005       0.001
Region_Centre                         -0.0044      0.002     -2.269      0.023      -0.008      -0.001
Region_Limousin                        0.0050      0.012      0.398      0.690      -0.019       0.029
Region_Nord-Pas-de-Calais              0.0039      0.005      0.772      0.440      -0.006       0.014
Region_Pays-de-la-Loire                0.0025      0.004      0.618      0.537      -0.005       0.010
Region_Poitou-Charentes                0.0019      0.005      0.356      0.722      -0.008       0.012
======================================================================================================
"""

In [17]:
selected_features_AIC_ClaimA

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Mercedes, Chrysler or BMW',
 'Brand_Opel, General Motors or Ford',
 'Brand_other',
 'Brand_Renault, Nissan or Citroen',
 'Brand_Volkswagen, Audi, Skoda or Seat',
 'Gas_Regular',
 'Region_Basse-Normandie',
 'Region_Bretagne',
 'Region_Centre',
 'Region_Haute-Normandie',
 'Region_Ile-de-France',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [18]:
selected_features_BIC_ClaimA

['Density',
 'Brand_Opel, General Motors or Ford',
 'Gas_Regular',
 'Region_Centre',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [19]:
# Ajouter une constante à X_test pour les deux modèles
X_test_const_AIC = sm.add_constant(X_test[selected_features_AIC_ClaimA])
X_test_const_BIC = sm.add_constant(X_test[selected_features_BIC_ClaimA])

# Prédictions pour les deux modèles
y_pred_test_AIC_ClaimA = final_model_AIC_ClaimA.predict(X_test_const_AIC)
y_pred_test_BIC_ClaimA = final_model_BIC_ClaimA.predict(X_test_const_BIC)

# Calcul des MSE pour les deux modèles
mse_AIC = mean_squared_error(y_test_ClaimA, y_pred_test_AIC_ClaimA)
mse_BIC = mean_squared_error(y_test_ClaimA, y_pred_test_BIC_ClaimA)
print("MSE pour le modèle AIC ClaimA :", mse_AIC)
print("MSE pour le modèle BIC ClaimA :", mse_BIC)

MSE pour le modèle AIC ClaimA : 77070358.48873481
MSE pour le modèle BIC ClaimA : 1929162.0829623197


In [33]:
print(y_pred_test_AIC_ClaimA.mean())
print(y_pred_test_BIC_ClaimA.mean())
print(y_test_ClaimA.mean())

78.87475363735635
86.42535681064042
73.71916127162045


0n remarque que les prediction de coût sont bonne en moyenne mais sous estime totalement la variance des sinistre réels qui sont au nombre de 3000 environ avec un cout moyen de 1900 ----> à REVOIR 